# Alibaba workload and RL experiment exploration

An interactive tour of the local Alibaba Cluster Trace 2017 sample and the saved v8 / topology-robust v9 experiments. Run cells from top to bottom. This notebook reads local files only; it does not download data or modify experiment results.

**Interpretation guardrails:** the trace supplies infrastructure arrivals and a derived estimate of compute work. MEC payloads, deadlines, device assignment, radios, batteries and capacities are simulation assumptions. The test split is one chronological window, the simulator is deterministic, and the 13 topology variants tested for v9 were also part of training. Robust v9 improves some cases, but does not universally beat Greedy EDF.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'pyproject.toml').exists()), HERE)
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))
print(f'Project root: {ROOT}')
print(f'Raw trace available: {(ROOT / "data" / "raw" / "batch_task.csv").exists()} and {(ROOT / "data" / "raw" / "batch_instance.csv").exists()}')

## 1. Data sources and provenance

The 2017 task file is headerless. Its eight fields are `create_timestamp, modify_timestamp, job_id, task_id, instance_num, status, plan_cpu, plan_mem`; the fifth field is the number of parallel instances, **not task size**. The loader joins valid completed instance records and estimates one task's demand as the mean instance runtime × `real_cpu_avg` × configured reference MIPS/core. It rejects multi-task jobs because their DAG dependencies are not reconstructed.

This adapter is versioned as `alibaba2017-mean-completed-instance-mi-v1`; do not compare models trained with the earlier summed-instance interpretation. Missing MEC fields are synthetic assumptions.

In [ ]:
def read_csv_if_present(relative_path):
    path = ROOT / relative_path
    if not path.exists():
        print(f'Missing optional result file: {path}')
        return pd.DataFrame()
    return pd.read_csv(path)

v8_aggregate = read_csv_if_present(Path('results') / 'alibaba-best-tier-calibrated-v8' / 'aggregate_metrics.csv')
v8_sweep = read_csv_if_present(Path('results') / 'alibaba-topology-sensitivity-v8' / 'topology_metrics.csv')
v9_sweep = read_csv_if_present(Path('results') / 'alibaba-topology-sensitivity-robust-v9' / 'topology_metrics.csv')
v8_log = read_csv_if_present(Path('artifacts') / 'models' / 'alibaba-best-tier-calibrated-v8' / 'training_log.csv')
v9_log = read_csv_if_present(Path('artifacts') / 'models' / 'alibaba-best-tier-calibrated-topology-robust-v9' / 'training_log.csv')
print('Rows:', {'v8 aggregate': len(v8_aggregate), 'v8 sweep': len(v8_sweep), 'v9 sweep': len(v9_sweep), 'v8 log': len(v8_log), 'v9 log': len(v9_log)})

## 2. Load and audit the raw trace (when available)

The repository intentionally does not distribute the raw Alibaba files. If either is missing, this section reports the expected paths and the saved-results analysis below still runs. Obtain the official 2017 archive under its terms, verify it using the upstream checksum, and place `batch_task.csv` and `batch_instance.csv` in `data/raw/`.

In [ ]:
task_path = ROOT / 'data' / 'raw' / 'batch_task.csv'
instance_path = ROOT / 'data' / 'raw' / 'batch_instance.csv'
tasks_df = pd.DataFrame()
splits = {}
if task_path.exists() and instance_path.exists():
    from mec_offloading.trace_workloads import TraceWorkloadConfig, load_alibaba_2017_batch_tasks, split_trace_tasks
    from mec_offloading.simulation import load_topology_config
    topology_path = ROOT / 'configs' / 'alibaba-multi-topology.yaml'
    sim_config = load_topology_config(topology_path)
    device_ids = tuple(d.device_id for d in sim_config.topology.devices)
    loaded = load_alibaba_2017_batch_tasks(
        task_path,
        instance_path=instance_path,
        config=TraceWorkloadConfig(max_tasks=None, device_ids=device_ids),
    )
    splits = split_trace_tasks(loaded.tasks)
    task_rows = []
    best_mips = max([d.cpu_capacity_mips for d in sim_config.topology.devices]
                    + [e.cpu_capacity_mips for e in sim_config.topology.edges]
                    + [sim_config.cloud_cpu_capacity_mips])
    for split_name, split_tasks in splits.items():
        for task in split_tasks:
            task_rows.append({
                'split': split_name, 'task_id': task.task_id, 'device_id': task.device_id,
                'arrival_time_s': task.arrival_time_s, 'required_mi': task.required_mi,
                'input_size_mb': task.input_size_mb, 'result_size_mb': task.result_size_mb,
                'deadline_s': max(1.0, 0.9 * task.required_mi / best_mips + 0.5),
            })
    tasks_df = pd.DataFrame(task_rows)
    print(f'Loaded {loaded.selected_count:,} tasks; rejected task rows: {loaded.rejected_count:,}; rejected auxiliary instance rows: {loaded.auxiliary_rejected_count:,}; stopped early: {loaded.stopped_early}')
    print('Task rejection reasons:', loaded.rejection_reasons)
    print('Split sizes:', {name: len(items) for name, items in splits.items()})
else:
    print('Raw Alibaba files not found. Expected:')
    print(' -', task_path)
    print(' -', instance_path)
    print('Continuing with saved training logs and experiment CSVs.')

## 3. Task-work distribution and chronological splits

The adapter's `required_mi` is a derived estimate. Log axes help reveal the long right tail; quantiles are reported alongside histograms because a few large tasks can dominate means. The configured deadline below is synthetic and uses the `best-tier-calibrated` policy from the training runs.

In [ ]:
if not tasks_df.empty:
    tasks_df['local_capacity_mips'] = tasks_df['device_id'].map({d.device_id: d.cpu_capacity_mips for d in sim_config.topology.devices})
    tasks_df['local_service_s'] = tasks_df['required_mi'] / tasks_df['local_capacity_mips']
    display(tasks_df.groupby('split').agg(task_count=('task_id', 'count'), arrival_min_s=('arrival_time_s', 'min'), arrival_max_s=('arrival_time_s', 'max'), median_work_mi=('required_mi', 'median'), median_local_s=('local_service_s', 'median')))
    display(tasks_df.groupby('split')[['required_mi', 'local_service_s', 'deadline_s']].quantile([.05, .25, .5, .75, .95, .99]).round(3))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for split_name, group in tasks_df.groupby('split', sort=False):
        axes[0].hist(group['required_mi'], bins=45, alpha=.5, label=split_name)
        axes[1].hist(group['local_service_s'], bins=np.logspace(-2, 4, 45), alpha=.5, label=split_name)
    axes[0].set(title='Derived task work', xlabel='Required work (MI)', ylabel='Tasks')
    axes[1].set(title='Isolated local service time', xlabel='Seconds (log scale)', ylabel='Tasks', xscale='log')
    for ax in axes: ax.legend()
    plt.tight_layout()
else:
    print('No raw task table to plot. Saved-run result exploration remains available.')

In [ ]:
if not tasks_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for split_name, group in tasks_df.groupby('split', sort=False):
        axes[0].hist(group['arrival_time_s'] / 3600, bins=40, alpha=.55, label=split_name)
    axes[0].set(title='Trace arrival times', xlabel='Hours from trace origin', ylabel='Tasks')
    axes[0].legend()
    train = tasks_df[tasks_df['split'] == 'train']
    axes[1].scatter(train['local_service_s'], train['deadline_s'], s=10, alpha=.45)
    axes[1].set(xscale='log', yscale='log', title='Training tasks: local time vs synthetic deadline', xlabel='Isolated local service (s)', ylabel='Synthetic deadline (s)')
    plt.tight_layout()
    print('Chronological windows are not shuffled; distribution shifts between splits are possible.')

## 4. Reference topology and model assumptions

This is a multi-device/multi-edge configuration, but the backend gives each device and edge one execution slot and uses one cloud worker. Shared-link contention is not modeled. Sensitivity variants are hypothetical parameter perturbations, not measured deployments.

In [ ]:
topology_file = ROOT / 'configs' / 'alibaba-multi-topology.yaml'
if topology_file.exists():
    import yaml
    topology_data = yaml.safe_load(topology_file.read_text(encoding='utf-8'))
    display(pd.DataFrame(topology_data['topology']['devices']).rename(columns={'id': 'device'}))
    display(pd.DataFrame(topology_data['topology']['edges']).rename(columns={'id': 'edge'}))
    print('Device-to-edge mapping:', topology_data['topology']['device_to_edge'])
    print('Cloud MIPS:', topology_data['resources']['cloud_cpu_mips'])
else:
    print('Topology YAML not found:', topology_file)

## 5. Held-out reference-topology policy comparison

This table is the saved v8 test run on 579 chronological test tasks (seed 10001). Raw success includes all tasks; feasible-task success conditions on the simulator's no-queue feasibility check. A small difference in one deterministic run is descriptive, not a significance test.

In [ ]:
if not v8_aggregate.empty:
    baseline = v8_aggregate[v8_aggregate['scenario'] == 'alibaba2017_test'].copy()
    baseline['raw_success_pct'] = 100 * baseline['deadline_success_rate']
    baseline['feasible_success_pct'] = 100 * baseline['feasible_deadline_success_rate']
    baseline['communication_GB'] = (baseline['communication_bytes'] + baseline['feedback_bytes']) / 1e9
    cols = ['policy', 'task_count', 'raw_success_pct', 'feasible_success_pct', 'median_latency_s', 'p95_latency_s', 'mean_device_energy_j', 'communication_GB']
    display(baseline[cols].round(3).sort_values('raw_success_pct', ascending=False))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(baseline['policy'], baseline['raw_success_pct'], color='#3973ac')
    ax.set(xlim=(0, 100), xlabel='Deadline success (%)', title='V8 held-out reference topology')
    ax.invert_yaxis()
    plt.tight_layout()
else:
    print('V8 aggregate metrics are missing; see reported path in the loader cell.')

## 6. Topology sensitivity: v8 vs robust v9

Both sweeps use the same test tasks and named variants. V9 trained on those same 13 variant types, so this is not an unseen-topology test. Interpret raw success with per-variant feasible-task success: for example, slow links make many deadlines infeasible even with empty queues.

In [ ]:
def select_hierarchical_periodic(frame, label):
    if frame.empty: return pd.DataFrame()
    out = frame[frame['policy'] == 'hierarchical_periodic'].copy()
    return out.set_index('variant').add_prefix(f'{label}_')

v8_h = select_hierarchical_periodic(v8_sweep, 'v8')
v9_h = select_hierarchical_periodic(v9_sweep, 'v9')
if not v8_h.empty and not v9_h.empty:
    comparison = v8_h.join(v9_h, how='inner')
    comparison['v8_raw_pct'] = 100 * comparison['v8_deadline_success_rate']
    comparison['v9_raw_pct'] = 100 * comparison['v9_deadline_success_rate']
    comparison['v8_feasible_pct'] = 100 * comparison['v8_feasible_deadline_success_rate']
    comparison['v9_feasible_pct'] = 100 * comparison['v9_feasible_deadline_success_rate']
    display(comparison[['v8_raw_pct', 'v9_raw_pct', 'v8_feasible_pct', 'v9_feasible_pct', 'v8_p95_latency_s', 'v9_p95_latency_s']].round(2))
    ax = comparison[['v8_raw_pct', 'v9_raw_pct']].plot(kind='bar', figsize=(14, 5), color=['#c0392b', '#2874a6'])
    ax.set(ylabel='Raw deadline success (%)', xlabel='Topology variant', title='Hierarchical periodic policy: v8 vs robust v9', ylim=(0, 105))
    ax.tick_params(axis='x', rotation=35)
    plt.tight_layout()
    ax = comparison[['v8_p95_latency_s', 'v9_p95_latency_s']].plot(kind='bar', figsize=(14, 5), color=['#c0392b', '#2874a6'])
    ax.set(ylabel='P95 latency (seconds)', xlabel='Topology variant', title='Tail latency under topology perturbations')
    ax.tick_params(axis='x', rotation=35)
    plt.tight_layout()
    greedy = v9_sweep[v9_sweep['policy'] == 'greedy_edf'].set_index('variant') if not v9_sweep.empty else pd.DataFrame()
    if not greedy.empty and 'baseline' in comparison.index:
        print(f"Reference topology raw success: robust hierarchical={comparison.loc['baseline', 'v9_raw_pct']:.1f}%; Greedy EDF={100*greedy.loc['baseline', 'deadline_success_rate']:.1f}%.")
else:
    print('One or both topology sweep CSVs are missing; comparison skipped.')

## 7. Training and validation curves

V8 checkpoint selection used reference-topology validation; robust v9 uses mean validation reward across seven topology variants. These curves have different validation objectives and should not be read as a direct head-to-head training-score comparison. Reward is negative cost, so closer to zero is better.

In [ ]:
def validation_rows(frame):
    if frame.empty: return pd.DataFrame()
    return frame[(frame['split'] == 'validation') & (frame['tier'] == 'hierarchical')].copy()

v8_val, v9_val = validation_rows(v8_log), validation_rows(v9_log)
if not v8_val.empty and not v9_val.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for label, frame, color in [('V8 reference validation', v8_val, '#c0392b'), ('V9 seven-topology validation', v9_val, '#2874a6')]:
        axes[0].plot(frame['episode'], frame['team_task_reward'], marker='o', label=label, color=color)
        axes[1].plot(frame['episode'], 100 * frame['feasible_deadline_success_rate'], marker='o', label=label, color=color)
    axes[0].set(xlabel='Episode', ylabel='Mean task reward', title='Validation reward (higher is better)')
    axes[1].set(xlabel='Episode', ylabel='Feasible-task success (%)', title='Validation feasibility-conditioned success')
    for ax in axes: ax.legend()
    plt.tight_layout()
else:
    print('V8 or v9 training log is missing; curve plotting skipped.')

## 8. Takeaways and limitations

- Fixing task semantics was foundational: a parallel stage must not be interpreted as one serial sum of every instance's work.
- In the reference topology, robust v9 hierarchical periodic success is about 85.7%, versus 86.0% for Greedy EDF; do not claim a general RL win.
- Topology augmentation notably improves the worst slow-device–edge-link raw-success result, but results are mixed, and p95 latency can worsen in that same case.
- One chronological trace split, one deterministic simulator test seed, fixed EDF cloud scheduling, one slot per device/edge, one cloud worker and no shared-link contention limit conclusions.
- Alibaba does not supply the MEC deadlines, payloads, assignments, radio, battery or topology values. These are assumptions, not trace measurements.

For experiment details, provenance, architecture and full result tables, see `docs/rl_experiment_report.md` and `docs/trace-data-notes.md`.